# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

**Entities are referenced by their `@id`.**

In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets and their Fields:")
record_sets = list(dataset.record_sets)
for record_set in record_sets:
    print(f"- Record Set Name: {record_set.name} | @id: {record_set.id}")
    fields = list(record_set.fields)
    for field in fields:
        dtype = getattr(field, 'data_type', '?')
        print(f"   - Field: {field.name} | @id: {field.id} | dataType: {dtype}")
    print()

### Preview first 2 records for each record set


In [ ]:
# Preview first 2 records
for record_set in record_sets:
    print(f"Sample records from Record Set: {record_set.name} | @id: {record_set.id}")
    for idx, record in enumerate(dataset.records(record_set=record_set.id)):
        print(record)
        if idx >= 1:
            break
    print('-'*50)

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract records for all available record sets into DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

# For demonstration, display the columns for the first record set
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    print(f"Columns of DataFrame for record set '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Choose a numeric field (e.g., coverage, peptide_count, MW) and a group field to illustrate. Replace the `numeric_field_id` and `group_field_id` as appropriate for your analysis.**

In [ ]:
# You may want to set these to relevant field @id values for your actual EDA
numeric_field_id = None
group_field_id = None
# Identify candidate fields
fields = list(record_sets[0].fields) if record_sets else []
for field in fields:
    if getattr(field, 'data_type', '').lower() in ['float', 'integer', 'number']:
        print(f"Numeric Field: {field.name} | @id: {field.id}")
        if not numeric_field_id:
            numeric_field_id = field.id
    if getattr(field, 'data_type', '').lower() in ['string', 'text', 'category']:
        print(f"Possible Group Field: {field.name} | @id: {field.id}")
        if not group_field_id:
            group_field_id = field.id

# EDA only if numeric field was found
if numeric_field_id and main_record_set_id:
    df = dataframes[main_record_set_id]
    # Filter to non-null/numeric only
    df = df.copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > Mean ({threshold:.2f}):")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group field (if exists)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. For example, plot the distribution of the chosen numeric field and a boxplot by group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if numeric_field_id and main_record_set_id:
    df = dataframes[main_record_set_id]
    df = df.copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If group field exists
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, showfliers=False)
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load, review, and analyze the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library, referencing all entities by their `@id`. The workflow included metadata inspection, record extraction per record set, basic exploratory analysis, and visualizations. For further analyses, see the field details under each record set and tailor the EDA to your research use case.